<a href="https://colab.research.google.com/github/Dill77/Birdcall_Individual_Clips_Audiocondenser/blob/main/Mpala_Audiocondenser_Individual_Clips.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#This program can simply run all at the start, get prompted with an audio file
#to upload, and then select desired audio file. Have BirdNET run and detect
#all species present, and splice out the audio into separate clips
#I highly recommend scanning any files through Chirpity first, as this is using the same
#background process via BirdNET, but takes significantly longer to analyse and cut all clips down

In [ ]:
# Bird Call Clip Extractor (application using BirdNET)

#This notebook takes an uploaded audio file, uses **BirdNET** (via the `birdnetlib` Python package) to detect bird vocalizations, and saves **each detected call as its own audio file**, named with the timestamp (from the original recording) at which it occurred.

#**How it works**
#1. Program will install all dependencies(BirdNET model + audio tools), so that (hopefully!) no other downloading needs to take place!
#2. Upload your audio file.
#3. Run BirdNET to detect bird calls with timestamps and confidence scores (level can be manually changed pre-upload!)
#4. Merge nearby detections and add a small padding so calls aren't cut short or repeated if calls have a small time lag in them
#5. Export each merged segment as its own file, named with its start/end timestamp
#6. Download all clips as a single zip.
# Quick note: using 'birdnetlib', which is just the python version of the same thing Chirpity uses - to get around access issues (and the fact that Chirpity doesn't have the function of cutting audio files!)

## 1. Install dependencies
#This takes a minute or two the first time you run it, but afterwards is faster
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q birdnetlib tensorflow pydub
print("Done installing dependencies.")

## 2. Upload your audio file
#Supports common formats (.wav in the case of our Mpala data)
from google.colab import files
import os

uploaded = files.upload()
assert len(uploaded) > 0, "No file uploaded."
INPUT_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {INPUT_PATH} ({os.path.getsize(INPUT_PATH)/1e6:.2f} MB)")

## 3. Configurable settings before running!

#- `MIN_CONFIDENCE`: is the minimum BirdNET confidence (0-1) for a detection to count as a real bird call. I think around 0.3 is reasonable, but can edit to be more or less tolerant!
#- `PADDING_SECONDS`: extra audio kept before/after each detected call so it isn't clipped abruptly. Change if audio has too much or too littel silence before and after (might be helpful to increase for ML training in the future)
#- `MERGE_GAP_SECONDS`: if two detections (after padding) are closer together than this, they're merged into one clip instead of being split apart for easier recognition by future models
#- `RECORDING_START`: Optional! As all files were recorded starting at a specific real-world date/time, user can set this (e.g. `datetime(2024, 5, 10, 6, 30, 0)`) and clip filenames will use corresponding real clock times. If left as `None`, filenames use offsets from the start of the file instead (e.g. `00-01-23_to_00-01-27`).
#- `LAT` / `LON` / `DATE`: optional. Giving BirdNET a location and date obviously improves detection accuracy. Leave as `None` to skip. Default should be the coordinates of Mpala Ranch - but currently bugged when changing location! DO NOT EDIT!!!

from datetime import datetime, timedelta

MIN_CONFIDENCE = 0.3
PADDING_SECONDS = 0.5
MERGE_GAP_SECONDS = 1.0

# Optional: real-world start time of the recording, for clock-time filenames.
# e.g. RECORDING_START = datetime(2024, 5, 10, 6, 30, 0)
RECORDING_START = None

# Optional, improves detection accuracy. Leave as None to skip. Currently bugged DO NOT EDIT!
LAT = None
LON = None
DATE = None     # e.g. datetime(2024, 5, 10)

## 4. Run BirdNET detection
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer

print("Loading BirdNET model (first run downloads model weights)...")
analyzer = Analyzer()

recording_kwargs = dict(min_conf=MIN_CONFIDENCE)
if LAT is not None and LON is not None:
    recording_kwargs["lat"] = LAT
    recording_kwargs["lon"] = LON
if DATE is not None:
    recording_kwargs["date"] = DATE

recording = Recording(analyzer, INPUT_PATH, **recording_kwargs)
print("Analyzing audio for bird calls...")
recording.analyze()

detections = recording.detections
print(f"Found {len(detections)} raw detections above confidence {MIN_CONFIDENCE}.")

for d in sorted(detections, key=lambda x: x['start_time'])[:10]:
    print(f"  {d['start_time']:.1f}s - {d['end_time']:.1f}s  {d['common_name']}  ({d['confidence']:.2f})")
if len(detections) > 10:
    print(f"  ... and {len(detections) - 10} more")

## 5. Merge detections into clip segments
def build_segments(detections, padding, merge_gap):
    if not detections:
        return []

    intervals = sorted(
        (max(0.0, d['start_time'] - padding), d['end_time'] + padding)
        for d in detections
    )

    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end + merge_gap:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged

segments = build_segments(detections, PADDING_SECONDS, MERGE_GAP_SECONDS)

print(f"{len(segments)} clip(s) will be produced.")
for start, end in segments:
    print(f"  clip {start:.1f}s - {end:.1f}s  ({end - start:.1f}s)")

## 6. Export each segment as its own file

#Filenames encode the timestamp of the clip within the original recording:
#- If `RECORDING_START` was set, filenames use real clock time, e.g. `birdcall_2024-05-10_06-31-23_to_06-31-27.wav`.
#- Otherwise, filenames use elapsed offset from the start of the file, e.g. `birdcall_00-01-23_to_00-01-27.wav` (hh-mm-ss).

from pydub import AudioSegment
import os, shutil

assert segments, "No bird calls detected above the confidence threshold -- nothing to extract. Try lowering MIN_CONFIDENCE."

audio = AudioSegment.from_file(INPUT_PATH)
duration_s = len(audio) / 1000.0
print(f"Original duration: {duration_s:.1f}s")

def offset_str(seconds):
    td = timedelta(seconds=int(seconds))
    total = int(td.total_seconds())
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}-{m:02d}-{s:02d}"

def clock_str(dt):
    return dt.strftime("%Y-%m-%d_%H-%M-%S")

OUTPUT_DIR = "bird_clips"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

clip_paths = []
for i, (start, end) in enumerate(segments, start=1):
    start_ms = int(max(0, start) * 1000)
    end_ms = int(min(duration_s, end) * 1000)
    clip = audio[start_ms:end_ms]

    if RECORDING_START is not None:
        clip_start_dt = RECORDING_START + timedelta(seconds=start)
        clip_end_dt = RECORDING_START + timedelta(seconds=end)
        tag = f"{clock_str(clip_start_dt)}_to_{clip_end_dt.strftime('%H-%M-%S')}"
    else:
        tag = f"{offset_str(start)}_to_{offset_str(end)}"

    filename = f"birdcall_{i:03d}_{tag}.wav"
    filepath = os.path.join(OUTPUT_DIR, filename)
    clip.export(filepath, format="wav")
    clip_paths.append(filepath)
    print(f"Saved: {filepath}  ({(end - start):.1f}s)")

print(f"\nExported {len(clip_paths)} clip(s) to '{OUTPUT_DIR}/'.")

## 7. Download all clips as a zip
import shutil
from google.colab import files as colab_files

zip_base = "bird_clips"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Zipped: {zip_path}")
colab_files.download(zip_path)


Done installing dependencies.


Saving Safari Cam-20250322-061137-1742613097198-7.mp4 to Safari Cam-20250322-061137-1742613097198-7.mp4
Uploaded: Safari Cam-20250322-061137-1742613097198-7.mp4 (704.64 MB)
Loading BirdNET model (first run downloads model weights)...
Labels loaded.
load model True
Model loaded.
Labels loaded.
load_species_list_model


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Meta model loaded.
Analyzing audio for bird calls...
read_audio_data


/usr/local/lib/python3.12/dist-packages/birdnetlib/main.py:310: UserWarning: PySoundFile failed. Trying audioread instead.
  self.ndarray, rate = librosa.load(
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


read_audio_data: complete, read  600 chunks.
analyze_recording Safari Cam-20250322-061137-1742613097198-7.mp4
Found 218 raw detections above confidence 0.3.
  3.0s - 6.0s  African Pied Wagtail  (0.68)
  15.0s - 18.0s  African Pied Wagtail  (0.94)
  18.0s - 21.0s  African Pied Wagtail  (0.91)
  21.0s - 24.0s  African Pied Wagtail  (0.77)
  30.0s - 33.0s  African Pied Wagtail  (0.43)
  33.0s - 36.0s  African Pied Wagtail  (0.69)
  36.0s - 39.0s  African Pied Wagtail  (0.35)
  39.0s - 42.0s  African Pied Wagtail  (0.90)
  42.0s - 45.0s  Von der Decken's Hornbill  (0.46)
  48.0s - 51.0s  African Pied Wagtail  (0.96)
  ... and 208 more
102 clip(s) will be produced.
  clip 2.5s - 6.5s  (4.0s)
  clip 14.5s - 24.5s  (10.0s)
  clip 29.5s - 45.5s  (16.0s)
  clip 47.5s - 66.5s  (19.0s)
  clip 68.5s - 75.5s  (7.0s)
  clip 77.5s - 84.5s  (7.0s)
  clip 86.5s - 93.5s  (7.0s)
  clip 98.5s - 105.5s  (7.0s)
  clip 107.5s - 111.5s  (4.0s)
  clip 113.5s - 117.5s  (4.0s)
  clip 119.5s - 123.5s  (4.0s)
  cl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print('--- Applying fixes ---')
!pip install --upgrade numba resampy
print("Upgraded numba and resampy.")

--- Applying fixes ---
Upgraded numba and resampy.


In [ ]:
# Ensure Analyzer is initialized (it should be from previous cell execution, but included for robustness!)
from birdnetlib.analyzer import Analyzer
from pydub import AudioSegment
import os, shutil
from datetime import datetime, timedelta # Ensure datetime and timedelta are imported here

# Function definitions from original notebook, moved here for chunking context
def offset_str(seconds):
    # timedelta is now imported at the top of the cell
    td = timedelta(seconds=int(seconds))
    total = int(td.total_seconds())
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}-{m:02d}-{s:02d}"

def build_segments(detections, padding, merge_gap):
    if not detections:
        return []

    intervals = sorted(
        (max(0.0, d['start_time'] - padding), d['end_time'] + padding)
        for d in detections
    )

    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end + merge_gap:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged


if 'analyzer' not in locals(): # Check if analyzer is not already defined
    print("Initializing BirdNET analyzer...")
    analyzer = Analyzer()

# Get full audio duration
full_audio = AudioSegment.from_file(INPUT_PATH)
full_duration_seconds = len(full_audio) / 1000.0
print(f"Total audio duration: {full_duration_seconds:.1f}s")

CHUNK_LENGTH_SECONDS = 10 * 60  # 10 minutes per chunk
all_detections_from_chunks = []
processed_clip_paths = [] # List to store paths of all processed clips

# Prepare output directory
OUTPUT_DIR = "bird_clips"
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

print(f"\nProcessing audio in {CHUNK_LENGTH_SECONDS/60:.0f}-minute chunks...")

for i, start_s in enumerate(range(0, int(full_duration_seconds), int(CHUNK_LENGTH_SECONDS))):
    end_s = min(start_s + CHUNK_LENGTH_SECONDS, full_duration_seconds)
    print(f"\n--- Processing chunk {i+1}: {offset_str(start_s)} to {offset_str(end_s)} ---")

    chunk_audio = full_audio[start_s * 1000 : end_s * 1000]
    chunk_filepath = os.path.join(OUTPUT_DIR, f"temp_chunk_{i:03d}.wav")
    chunk_audio.export(chunk_filepath, format="wav")

    # Run BirdNET on the chunk
    chunk_recording = Recording(analyzer, chunk_filepath, **recording_kwargs)
    print(f"Analyzing chunk {i+1}...")
    chunk_recording.analyze()

    # Adjust detection timestamps to original file's time and collect
    for d in chunk_recording.detections:
        d['start_time'] += start_s
        d['end_time'] += start_s
        all_detections_from_chunks.append(d)

    # Clean up temporary chunk file
    os.remove(chunk_filepath)

print(f"\nFound {len(all_detections_from_chunks)} raw detections across all chunks above confidence {MIN_CONFIDENCE}.")

# Assign to 'detections' variable for compatibility with subsequent merging logic
detections = all_detections_from_chunks

Total audio duration: 1800.0s

Processing audio in 10-minute chunks...

--- Processing chunk 1: 00-00-00 to 00-10-00 ---
Analyzing chunk 1...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_000.wav

--- Processing chunk 2: 00-10-00 to 00-20-00 ---
Analyzing chunk 2...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_001.wav

--- Processing chunk 3: 00-20-00 to 00-30-00 ---
Analyzing chunk 3...
read_audio_data
read_audio_data: complete, read  200 chunks.
analyze_recording temp_chunk_002.wav

Found 218 raw detections across all chunks above confidence 0.3.


In [ ]:
from datetime import datetime, timedelta # Ensure datetime and timedelta are imported here

print(f"\nMerging {len(detections)} detections and exporting clips...")

# Merge detections from all chunks into final segments
sesgments = build_segments(detections, PADDING_SECONDS, MERGE_GAP_SECONDS)

print(f"{len(segments)} final clip(s) will be produced.")

assert segments, "No bird calls detected above the confidence threshold after chunking -- nothing to extract. Try lowering MIN_CONFIDENCE."

# Adding clock_str definition from original notebook for this cell's context
def clock_str(dt):
    # datetime is now imported at the top of the cell
    return dt.strftime("%Y-%m-%d_%H-%M-%S")

for i, (start, end) in enumerate(segments, start=1):
    start_ms = int(max(0, start) * 1000)
    end_ms = int(min(full_duration_seconds, end) * 1000) # Use full_duration_seconds
    clip = full_audio[start_ms:end_ms]

    if RECORDING_START is not None:
        # timedelta is now imported at the top of the cell
        clip_start_dt = RECORDING_START + timedelta(seconds=start)
        clip_end_dt = RECORDING_START + timedelta(seconds=end)
        tag = f"{clock_str(clip_start_dt)}_to_{clip_end_dt.strftime('%H-%M-%S')}"
    else:
        tag = f"{offset_str(start)}_to_{offset_str(end)}"

    filename = f"birdcall_{i:03d}_{tag}.wav"
    filepath = os.path.join(OUTPUT_DIR, filename)
    clip.export(filepath, format="wav")
    processed_clip_paths.append(filepath) # Use the new list
    print(f"Saved: {filepath}  ({(end - start):.1f}s)")

print(f"\nExported {len(processed_clip_paths)} clip(s) to '{OUTPUT_DIR}/'.")


Merging 218 detections and exporting clips...
102 final clip(s) will be produced.
Saved: bird_clips/birdcall_001_00-00-02_to_00-00-06.wav  (4.0s)
Saved: bird_clips/birdcall_002_00-00-14_to_00-00-24.wav  (10.0s)
Saved: bird_clips/birdcall_003_00-00-29_to_00-00-45.wav  (16.0s)
Saved: bird_clips/birdcall_004_00-00-47_to_00-01-06.wav  (19.0s)
Saved: bird_clips/birdcall_005_00-01-08_to_00-01-15.wav  (7.0s)
Saved: bird_clips/birdcall_006_00-01-17_to_00-01-24.wav  (7.0s)
Saved: bird_clips/birdcall_007_00-01-26_to_00-01-33.wav  (7.0s)
Saved: bird_clips/birdcall_008_00-01-38_to_00-01-45.wav  (7.0s)
Saved: bird_clips/birdcall_009_00-01-47_to_00-01-51.wav  (4.0s)
Saved: bird_clips/birdcall_010_00-01-53_to_00-01-57.wav  (4.0s)
Saved: bird_clips/birdcall_011_00-01-59_to_00-02-03.wav  (4.0s)
Saved: bird_clips/birdcall_012_00-02-05_to_00-02-09.wav  (4.0s)
Saved: bird_clips/birdcall_013_00-02-11_to_00-02-15.wav  (4.0s)
Saved: bird_clips/birdcall_014_00-02-17_to_00-02-21.wav  (4.0s)
Saved: bird_clips/